# JEPA: Systematic Study

Cross-tick latent prediction. Excels on visual tasks (+9pp at w=0.1).

| tier | what | runs |
|---|---|---|
| **Main** | baseline vs JEPA(0.1), 5 seeds x 2 tasks | 10 |
| **Sweep** | weight {0.02..0.5}, 3 seeds x 2 tasks | 36 |
| **Ablation** | loss/stopgrad/depth/hidden, 3 seeds x 2 tasks | 42 |
| **Total** | | **88** |

**Hardware**: 1 machine x 8 GPUs (~12h).

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

## Part A - Prior Results (st04 jepa_w0.1)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    print(summary_stats(df_prior[(df_prior.stage=='st04') & (df_prior.sweep=='jepa_w0.1')]))
else:
    print('Prior data not found.')

In [ ]:
if df_prior is not None:
    plot_prior_bar(df_prior, ['cifar10','mazes'],
                   'st04', 'jepa_w0.1', 'Prior: JEPA w=0.1 vs baseline',
                   'figures/02_prior_bar.png')

In [ ]:
curves = load_prior_curves()
if curves:
    plot_prior_curves(curves, 'cifar10',
        [('st00','paper','baseline','#888'),
         ('st04','jepa_w0.1','JEPA 0.1','#1f77b4')],
        'cifar10 convergence (prior)', 'figures/02_prior_conv.png')

## Part B - Experiment Design

In [ ]:
main_exps = make_jepa(['cifar10','mazes'], [0,1,2,3,4], weights=[0.1])
sweep_exps = make_jepa_sweep(['cifar10','mazes'], [0,1,2])
ablation_exps = make_jepa_ablation(['cifar10','mazes'], [0,1,2])
exps = main_exps + sweep_exps + ablation_exps
print(f'Main: {len(main_exps)}, Sweep: {len(sweep_exps)}, Ablation: {len(ablation_exps)}')
print(f'Total: {len(exps)} experiments')

In [ ]:
run_all(exps[:5], gpus=8, log_root='logs/deep/02_jepa', dry_run=True)
print('...')
run_all(exps[-5:], gpus=8, log_root='logs/deep/02_jepa', dry_run=True)

### Run all (~12h on 8 GPUs)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/02_jepa')

In [ ]:
status('logs/deep/02_jepa')

## Part C - Main Results

In [ ]:
df = collect('logs/deep/02_jepa')
if df.empty:
    print('No results yet.')
else:
    df_main = df[df.name.str.contains('jepa_w0p1_s') & ~df.name.str.contains('swp|abl')]
    if not df_main.empty:
        print(df_main[['name','task','best_acc','delta']].to_string(index=False))
        plot_delta_bars(df_main, 'Main: JEPA(0.1) vs baseline (5 seeds)',
                        'figures/02_main_delta.png')

In [ ]:
if not df.empty:
    df_main = df[df.name.str.contains('jepa_w0p1_s') & ~df.name.str.contains('swp|abl')]
    if not df_main.empty:
        sig = significance_test(df_main)
        print(sig.to_string(index=False))

## Part D - Weight Sensitivity

In [ ]:
if not df.empty:
    import re
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        df_sw['weight'] = df_sw['name'].str.extract(r'w([0-9]+p?[0-9]*)_s')[0].str.replace('p','.').astype(float)
        plot_sweep_heatmap(df_sw, 'weight', 'task',
                          'JEPA: weight x task delta (pp)',
                          'figures/02_sweep_heatmap.png')

In [ ]:
if not df.empty:
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        import re
        df_sw['weight'] = df_sw['name'].str.extract(r'w([0-9]+p?[0-9]*)_s')[0].str.replace('p','.').astype(float)
        plot_sweep_curve(df_sw, 'weight',
                        'JEPA weight sweep (errorbar = std over seeds)',
                        'figures/02_sweep_curve.png')

## Part E - Ablation Study

In [ ]:
if not df.empty:
    df_abl = df[df.name.str.contains('abl_')].copy()
    if not df_abl.empty:
        import re
        df_abl['variant'] = df_abl['name'].str.extract(r'abl_([a-z_0-9]+)_s')[0]
        plot_ablation_bars(df_abl, 'variant',
                          'JEPA ablation: component contributions',
                          'figures/02_ablation.png')

In [ ]:
if not df.empty:
    df_abl = df[df.name.str.contains('abl_')].copy()
    if not df_abl.empty:
        import re
        df_abl['variant'] = df_abl['name'].str.extract(r'abl_([a-z_0-9]+)_s')[0]
        print(summary_stats(df_abl, groupby=('task','variant')))